In [1]:
%pip install plotly

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors

In [3]:
df = pd.read_csv("ecommerce_data.csv")


In [4]:
df.head()

,CustomerID,InvoiceDate,Quantity,UnitPrice,TotalAmount,Country,ProductCategory
0,C0045,2022-08-24 00:00:00,2,39.69,79.39,UK,Books
1,C0757,2022-07-01 00:00:00,1,414.09,414.09,Netherlands,Sports
2,C1643,2022-01-05 00:00:00,1,358.44,358.44,NaN,Sports
3,C0133,2023-07-01 00:00:00,1,150.21,150.21,UK,Clothing
4,C0941,2022-12-10 00:00:00,1,222.65,222.65,UK,Sports


In [5]:
df[df["UnitPrice"]==0]

,CustomerID,InvoiceDate,Quantity,UnitPrice,TotalAmount,Country,ProductCategory
184,C0566,2022-04-21 00:00:00,1,0.0,672.46,UK,Electronics
772,C0672,2022-03-31 00:00:00,1,0.0,17.09,Germany,Books
840,C0137,2023-03-03 00:00:00,1,0.0,371.92,NaN,Sports
964,C1644,2022-06-04 00:00:00,1,0.0,269.02,UK,Electronics
1221,C1631,2022-04-22 00:00:00,1,0.0,29.52,UK,Home & Garden
2615,C1052,2022-06-02 00:00:00,1,0.0,463.60,Germany,Electronics
3577,C0231,2022-10-06 00:00:00,1,0.0,24.04,Spain,Books
3639,C1048,2022-03-20 00:00:00,4,0.0,216.76,UK,Beauty
3970,C1844,2022-04-28 00:00:00,1,0.0,36.28,Germany,Home & Garden
4273,C1018,2022-05-21 00:00:00,1,0.0,105.27,NaN,Clothing


In [6]:
df["Quantity"] = abs(df["Quantity"])


In [7]:
df["unit_price"] = df["TotalAmount"]/df["Quantity"]

In [8]:
df.drop("UnitPrice", axis=1, inplace=True)

In [ ]:
df.head()

,CustomerID,InvoiceDate,Quantity,TotalAmount,Country,ProductCategory,unit_price
0,C0045,2022-08-24 00:00:00,2,79.39,UK,Books,39.695
1,C0757,2022-07-01 00:00:00,1,414.09,Netherlands,Sports,414.090
2,C1643,2022-01-05 00:00:00,1,358.44,NaN,Sports,358.440
3,C0133,2023-07-01 00:00:00,1,150.21,UK,Clothing,150.210
4,C0941,2022-12-10 00:00:00,1,222.65,UK,Sports,222.650


In [10]:
df.dropna(subset=["CustomerID"], inplace=True)

In [12]:
(16/df.shape[0])*100

0.19538405177677373

In [13]:
df[df["Country"].isnull()]

,CustomerID,InvoiceDate,Quantity,TotalAmount,Country,ProductCategory,unit_price
2,C1643,2022-01-05 00:00:00,1,358.44,NaN,Sports,358.44
14,C1802,2022-11-12 00:00:00,1,602.07,NaN,Electronics,602.07
18,C0109,2022-07-31 00:00:00,1,96.24,NaN,Home & Garden,96.24
24,C0902,2022-10-31 00:00:00,1,42.46,NaN,Beauty,42.46
29,C0067,2022-11-22 00:00:00,1,219.06,NaN,Electronics,219.06
...,...,...,...,...,...,...,...
8182,C1676,2023-02-06 00:00:00,1,339.53,NaN,Electronics,339.53
8185,C0664,2022-12-25 00:00:00,1,40.21,NaN,Beauty,40.21
8192,C1411,2022-07-19 00:00:00,1,562.77,NaN,Sports,562.77
8196,C1498,2023-02-14 00:00:00,1,470.56,NaN,Sports,470.56


In [14]:
df.info()

<class 'pandas.DataFrame'>
Index: 8189 entries, 0 to 8204
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerID       8189 non-null   str    
 1   InvoiceDate      8189 non-null   str    
 2   Quantity         8189 non-null   int64  
 3   TotalAmount      8189 non-null   float64
 4   Country          6851 non-null   str    
 5   ProductCategory  8189 non-null   str    
 6   unit_price       8189 non-null   float64
dtypes: float64(2), int64(1), str(4)
memory usage: 804.9 KB


In [15]:
# Step 1: build the lookup
lookup = df.dropna(subset=["Country", "CustomerID"]).drop_duplicates("CustomerID").set_index("CustomerID")["Country"]

# Step 2: count missing before filling
missing = df["Country"].isnull()
total_missing = missing.sum()

# Step 3: fill from the lookup
df.loc[missing, "Country"] = df.loc[missing, "CustomerID"].map(lookup)

# Step 4: count what is still missing after the lookup
still_missing = df["Country"].isnull().sum()
filled = total_missing - still_missing

# Step 5: label the rest as Unknown
df["Country"] = df["Country"].fillna("Unknown")

# Results
print("Originally missing:", total_missing)
print("Filled from CustomerID:", filled)
print("Labelled Unknown:", still_missing)

Originally missing: 1338
Filled from CustomerID: 0
Labelled Unknown: 1338


In [16]:
unk = df["Country"] == "Unknown"

print("Unknown rows:", unk.sum())
print("...with CustomerID null:", df.loc[unk, "CustomerID"].isnull().sum())
print("...with CustomerID present:", df.loc[unk, "CustomerID"].notnull().sum())

# of those with an ID, how many IDs exist in rows that have a known country?
known_ids = df.loc[~unk, "CustomerID"].dropna().unique()
print("...whose ID appears with a known country:", df.loc[unk, "CustomerID"].dropna().isin(known_ids).sum())

print(df["CustomerID"].dtype)

Unknown rows: 1338
...with CustomerID null: 0
...with CustomerID present: 1338
...whose ID appears with a known country: 0
str


In [17]:
unk = df["Country"] == "Unknown"

print(df.loc[unk, "CustomerID"].head(5).tolist())
print(df.loc[~unk, "CustomerID"].head(5).tolist())
print("Unique customers in Unknown group:", df.loc[unk, "CustomerID"].nunique())

['C1643', 'C1802', 'C0109', 'C0902', 'C0067']
['C0045', 'C0757', 'C0133', 'C0941', 'C0802']
Unique customers in Unknown group: 281


In [18]:
df.info()

<class 'pandas.DataFrame'>
Index: 8189 entries, 0 to 8204
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerID       8189 non-null   str    
 1   InvoiceDate      8189 non-null   str    
 2   Quantity         8189 non-null   int64  
 3   TotalAmount      8189 non-null   float64
 4   Country          8189 non-null   str    
 5   ProductCategory  8189 non-null   str    
 6   unit_price       8189 non-null   float64
dtypes: float64(2), int64(1), str(4)
memory usage: 813.0 KB


In [19]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [20]:
df.head()

,CustomerID,InvoiceDate,Quantity,TotalAmount,Country,ProductCategory,unit_price
0,C0045,2022-08-24,2,79.39,UK,Books,39.695
1,C0757,2022-07-01,1,414.09,Netherlands,Sports,414.090
2,C1643,2022-01-05,1,358.44,Unknown,Sports,358.440
3,C0133,2023-07-01,1,150.21,UK,Clothing,150.210
4,C0941,2022-12-10,1,222.65,UK,Sports,222.650


In [21]:
reference_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
reference_date

Timestamp('2024-01-01 00:00:00')

In [22]:
rfm= df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (reference_date - x.max()).days),
    Frequency=("TotalAmount", "count"),
    Monetary=("TotalAmount", "sum")).reset_index()


In [23]:
rfm

,CustomerID,Recency,Frequency,Monetary
0,C0001,471,4,421.60
1,C0002,57,14,3038.91
2,C0003,282,8,1200.31
3,C0004,295,2,136.24
4,C0005,618,1,68.88
...,...,...,...,...
1842,C1846,206,10,2211.74
1843,C1847,503,3,617.96
1844,C1848,615,1,14.41
1845,C1849,720,1,101.92


In [24]:
# Calculate RFM Scores
rfm = rfm.set_index("CustomerID")
quantiles = rfm.quantile(q=[0.25, 0.5,0.75])

# define a function to assign RFM scores based on quantiles
def RScore(x,p,d):
    if p == 'Recency':
        if x <= d[p][0.25]:
            return 4
        elif x <= d[p][0.50]:
            return 3
        elif x <= d[p][0.75]:
            return 2
        else:
            return 1
    else:
        if x <= d[p][0.25]:
            return 1
        elif x <= d[p][0.50]:
            return 2
        elif x <= d[p][0.75]:
            return 3
        else:
            return 4

rfm['R'] = rfm['Recency'].apply(RScore, args=('Recency',quantiles,))
rfm['F'] = rfm['Frequency'].apply(RScore, args=('Frequency',quantiles,))
rfm['M'] = rfm['Monetary'].apply(RScore, args=('Monetary',quantiles,))

In [25]:
rfm

,Recency,Frequency,Monetary,R,F,M
CustomerID,,,,,,
C0001,471,4,421.60,2,3,2
C0002,57,14,3038.91,4,4,4
C0003,282,8,1200.31,4,4,3
C0004,295,2,136.24,4,2,1
C0005,618,1,68.88,1,1,1
...,...,...,...,...,...,...
C1846,206,10,2211.74,4,4,4
C1847,503,3,617.96,2,3,3
C1848,615,1,14.41,1,1,1


In [26]:
# rfm['RFM_Segment'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
rfm['RFM_Score'] = rfm[['R','F','M']].sum(axis=1)

In [27]:
rfm

,Recency,Frequency,Monetary,R,F,M,RFM_Score
CustomerID,,,,,,,
C0001,471,4,421.60,2,3,2,7
C0002,57,14,3038.91,4,4,4,12
C0003,282,8,1200.31,4,4,3,11
C0004,295,2,136.24,4,2,1,7
C0005,618,1,68.88,1,1,1,3
...,...,...,...,...,...,...,...
C1846,206,10,2211.74,4,4,4,12
C1847,503,3,617.96,2,3,3,8
C1848,615,1,14.41,1,1,1,3


In [28]:
segment_labels = ['low-value', 'mid-value', 'high-value']
def assign_segment(score):
    if score < 5:
        return 'low-value'
    elif score < 9:
        return 'mid-value'
    else:
        return 'high-value'

rfm['RFM_Segment'] = rfm['RFM_Score'].apply(assign_segment)
rfm.head()

,Recency,Frequency,Monetary,R,F,M,RFM_Score,RFM_Segment
CustomerID,,,,,,,,
C0001,471,4,421.60,2,3,2,7,mid-value
C0002,57,14,3038.91,4,4,4,12,high-value
C0003,282,8,1200.31,4,4,3,11,high-value
C0004,295,2,136.24,4,2,1,7,mid-value
C0005,618,1,68.88,1,1,1,3,low-value


In [29]:
# Count the number of customers in each RFM segment
segment_counts = rfm['RFM_Segment'].value_counts().reset_index()
segment_counts.columns = ['RFM_Segment', 'Count']
segment_counts = segment_counts.sort_values('RFM_Segment')

In [30]:
fig = px.bar(segment_counts,
             x='RFM_Segment',
             y='Count',
             color='RFM_Segment',
             title='Customer Distribution based on RFM Segment',
             labels={'RFM_Segment': 'Customer Segment',
                     'Count': 'Number of Customers'},
color_discrete_sequence=px.colors.qualitative.Pastel)

fig.show()

In [31]:
rfm['RFM_Customer_Segment'] = ''

rfm.loc[rfm['RFM_Score'] >= 9, 'RFM_Customer_Segment'] = 'VIP'
rfm.loc[(rfm['RFM_Score'] >= 6) & (rfm['RFM_Score'] < 9), 'RFM_Customer_Segment'] = 'Potential Loyal'
rfm.loc[rfm['RFM_Score'] == 5, 'RFM_Customer_Segment'] = 'At Risk'
rfm.loc[rfm['RFM_Score'] == 4, 'RFM_Customer_Segment'] = "Can't Lose Them"
rfm.loc[rfm['RFM_Score'] <= 3, 'RFM_Customer_Segment'] = 'Lost'

print(rfm['RFM_Customer_Segment'].value_counts())

RFM_Customer_Segment
VIP                720
Potential Loyal    462
Can't Lose Them    258
At Risk            224
Lost               183
Name: count, dtype: int64


In [32]:
rfm['RFM_Customer_Segment'].value_counts()

RFM_Customer_Segment
VIP                720
Potential Loyal    462
Can't Lose Them    258
At Risk            224
Lost               183
Name: count, dtype: int64

In [33]:
# group the data by RFM_Segment and RFM_Customer_Segment, and count the number of customers in each segment
segment_product_counts = rfm.groupby(['RFM_Segment', 'RFM_Customer_Segment']).size().reset_index(name='Count')

segment_product_counts = segment_product_counts.sort_values('Count', ascending=False)

In [34]:
fig_treemap_segment_product = px.treemap(segment_product_counts,
                                         path=['RFM_Segment', 'RFM_Customer_Segment'],
                                         values='Count',
                                         color= 'RFM_Segment',
                                         color_discrete_sequence=px.colors.qualitative.Pastel,
                                         title='Customer Distribution based on Value')

fig_treemap_segment_product.show()

In [35]:
vips= rfm[rfm['RFM_Customer_Segment'] == 'VIP']

In [36]:
fig = go.Figure()
fig.add_trace(go.Box(y=vips['Recency'], name = 'Recency'))
fig.add_trace(go.Box(y=vips['Frequency'], name = 'Frequency'))
fig.add_trace(go.Box(y=vips['Monetary'], name = 'Monetary'))

In [37]:
correlation_matrix = vips[['R', 'F','M']].corr()

fig_heatmap = go.Figure(data = go.Heatmap(z=correlation_matrix.values,
                                          x=correlation_matrix.columns,
                                          y=correlation_matrix.index,
                                          colorbar=dict(title= 'Correlation'))
)

fig_heatmap.update_layout(title='Correlation Heatmap for VIP Customers')
# show the heatmap
fig_heatmap.show()

In [38]:
rfm

,Recency,Frequency,Monetary,R,F,M,RFM_Score,RFM_Segment,RFM_Customer_Segment
CustomerID,,,,,,,,,
C0001,471,4,421.60,2,3,2,7,mid-value,Potential Loyal
C0002,57,14,3038.91,4,4,4,12,high-value,VIP
C0003,282,8,1200.31,4,4,3,11,high-value,VIP
C0004,295,2,136.24,4,2,1,7,mid-value,Potential Loyal
C0005,618,1,68.88,1,1,1,3,low-value,Lost
...,...,...,...,...,...,...,...,...,...
C1846,206,10,2211.74,4,4,4,12,high-value,VIP
C1847,503,3,617.96,2,3,3,8,mid-value,Potential Loyal
C1848,615,1,14.41,1,1,1,3,low-value,Lost


In [39]:
segment_counts = rfm['RFM_Customer_Segment'].value_counts()
pastel_colors =plotly.colors.qualitative.Pastel
fig = go.Figure(data = [
    go.Bar(x= segment_counts.index, y=segment_counts.values, marker=dict(color=pastel_colors))])

vip_color = 'rgb(158, 202, 225)'
fig.update_traces(marker_color=[vip_color if segment == 'VIP' else pastel_colors[i] for i, segment in enumerate(segment_counts.index)], marker_line_color='rgb(8,48,107)', marker_line_width=1.5, opacity=0.6)

fig.update_layout(title='Customer Distribution based on RFM Segment',
                  xaxis_title='RFM Segment',
                  yaxis_title='Number of Customers',
                  showlegend=False)

fig.show()